# Step 7: metrics and robustness checks of all the variants (train only)

**Question.** Are the differences between the variants and their reference in Step 6 ([06_backtest.ipynb](06_backtest.ipynb)) real or luck? For each variant:

1. final table: full metrics, difference from the reference with a confidence interval (bootstrap), beta and alpha against SPY;
2. sensitivity to costs;
3. dependence on a sector or a pair;
4. dependence on the period: the two halves of the train and the train without 13 August (the CRM-WDAY day).

Main scenario: cost 0. **Train only**: the test was not read.

**What is not done here.** The sensitivity to nearby signal parameters (thresholds, half-life) is in Step 4 for A. For the variants it cannot be done without running the agents again, because their decisions hold only for the signals with the chosen parameters.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # project folder: makes src importable

import pandas as pd

from src.config import RESULTS
from src.backtest import load_trades, trading_days, add_costs, daily_pnl, summary, save_results, CAPITAL_PER_PAIR
from src.agent_variants.comparison import references, all_variants, daily_delta, bootstrap, market, beta_alpha, sectors

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
trades, days = load_trades(), trading_days()          # train only
ref = references(trades)
variants = all_variants(trades)
mkt = market(days)
key = lambda v: (v.name, v.version)                   # noqa: E731
print(f"{len(variants)} variants, {len(days)} train days; SPY: mean return {mkt.mean() * 252:.1%} a year, volatility {mkt.std() * 252 ** 0.5:.1%}")

## 1. Final table (cost 0)

- metrics of each variant and of the references;
- `delta_usd`: dollar difference from the reference, with the 95% confidence interval obtained by resampling the 64 days with replacement (10,000 times);
- `share_below_zero`: the share of resamples in which the variant earns less than the reference;
- beta and annualised alpha against SPY, with their t statistics. SPY: daily prices adjusted for dividends, downloaded from Alpaca (`python -m src.spy`).

The bootstrap treats the days as independent.

In [ ]:
def metrics(t):
    s = summary(add_costs(t, 0), days)
    return {k: s[k] for k in ("trade", "net_bps", "win_rate", "profit_factor", "avg_minutes", "pnl_usd", "sharpe")} | dict(max_drawdown_pct=100 * s["max_drawdown"]) | \
           beta_alpha(daily_pnl(add_costs(t, 0), days).ret, mkt)

rows = [dict(variant=n, version="", reference="") | metrics(t) for n, t in ref.items()]
rows += [dict(variant=v.name, version=v.version, reference=v.reference) | metrics(v.trades) | bootstrap(daily_delta(v, ref, days))
         for v in variants]
final = pd.DataFrame(rows).set_index(["variant", "version"])
final = pd.concat([final.iloc[:len(ref)], final.iloc[len(ref):].sort_values("delta_usd", ascending=False)])
final[["reference", "trade", "net_bps", "win_rate", "profit_factor", "sharpe", "max_drawdown_pct", "pnl_usd", "delta_usd", "ci95_low", "ci95_high",
       "share_below_zero", "beta", "t_beta", "alpha_annual", "t_alpha"]]

## 2. Sensitivity to costs

Dollar difference from the reference with 0, 2, 5 and 10 basis points per operation. Every blocked trade saves its costs: with high costs even a random block improves the result, so an advantage that shows up only at a high cost does not mean that the agent chooses well.

In [ ]:
COSTS = (0, 2, 5, 10)
costs = pd.DataFrame({key(v): {c: summary(add_costs(v.trades, c), days)["pnl_usd"] - summary(add_costs(ref[v.reference], c), days)["pnl_usd"]
                               for c in COSTS} for v in variants}).T.rename_axis(["variant", "version"])
costs.columns = [f"delta_usd_cost_{c}" for c in COSTS]
costs.sort_values("delta_usd_cost_0", ascending=False)

## 3. Does it depend on a sector or a pair?

Dollar difference from the reference (cost 0) per sector. Then, for each variant, the pair that weighs most on the difference and the difference without that pair.

In [ ]:
sector = sectors()

def delta_by_pair(v):
    p = lambda t: add_costs(t, 0).groupby("pair").pnl_usd.sum()   # noqa: E731
    return p(v.trades).sub(p(ref[v.reference]), fill_value=0).reindex(sector.index, fill_value=0)

by_pair = pd.DataFrame({key(v): delta_by_pair(v) for v in variants}).T.rename_axis(["variant", "version"])
by_pair.T.groupby(sector).sum().T.loc[costs.sort_values("delta_usd_cost_0", ascending=False).index]

In [ ]:
top = by_pair.abs().idxmax(axis=1)
pd.DataFrame({"delta_usd": by_pair.sum(axis=1), "heaviest_pair": top,
              "delta_of_that_pair": [by_pair.loc[i, p] for i, p in top.items()],
              "delta_without_that_pair": [by_pair.loc[i].drop(p).sum() for i, p in top.items()]}).sort_values("delta_usd", ascending=False)

## 4. Does it depend on the period?

Dollar difference from the reference (cost 0) in the two halves of the train (32 days each) and on the whole train without 13 August, the day of the WDAY jump (Step 4).

In [ ]:
half = days[len(days) // 2]
period = pd.DataFrame({key(v): (lambda d: {"first_half": d[d.index < half].sum(), "second_half": d[d.index >= half].sum(),
                                           "without_13_august": d.drop(pd.Timestamp("2026-08-13")).sum(), "all": d.sum()})(daily_delta(v, ref, days))
                       for v in variants}).T.rename_axis(["variant", "version"])
period.sort_values("all", ascending=False)

## 5. Saving

In [ ]:
for p in save_results({"robustness/final_table": final, "robustness/costs": costs, "robustness/pairs": by_pair,
                       "robustness/sectors": by_pair.T.groupby(sector).sum().T, "robustness/period": period}):
    print(p.relative_to(RESULTS.parent))

## 6. What the checks tell us

**No difference stands out from chance.**
- For almost all the variants the 95% interval of the difference from the reference contains zero.
- The exceptions are the C of `lexicon` (from -4,600 to -499 dollars) and `spark24h_agent_size` (from -5,740 to -1,220): they are worse than A with some confidence.
- Even `b_exaggeration`, the only one above A, has an interval from -204 to +863 dollars: it is below A in 13% of the resamples.

**Market neutral.** Against SPY all the variants have a practically zero beta (from -0.01 to +0.01, t statistic below 1.2), as expected from pairs trading: the profit does not come from the direction of the stock market. The annualised alpha of A (9.5%, t = 4.1) must be read with caution: it is 64 days scaled to a year.

**Costs.** With high costs many B beat A (`lexicon` B +5,113 dollars at 10 bps) only because they make fewer trades. It is a cost saving, not a better choice: the same B lose at cost 0.

**Pairs and period.** The ranking depends on one or two pairs and on a few days.
- **FinBERT variants:** the difference comes almost entirely from PYPL-XYZ. Without that pair, `finbert_session` B goes from -469 to +95 dollars, `finbert_episode` B from -285 to +279, and the delayed B become positive (+380 and +443). Moreover they lose in the first half of the train and earn in the second.
- **Giulio's variants:** CRM-WDAY weighs. Without 13 August, the B of `spark_24h` loses 395 dollars instead of 1,334.
- **`b_exaggeration`:** it is above A in both halves, but its rule was chosen on these data.

**The 3 candidates and the chosen variant.** The candidates and the choice of `finbert_episode` are motivated in Step 6. The checks above say:

| B | Difference from the reference | 95% interval | Without PYPL-XYZ | First half / second half |
|---|---|---|---|---|
| `finbert_delay_15min` | -63 | from -1,086 to +745 | +443 | -302 / +239 |
| `finbert_delay_5min` | -75 | from -959 to +809 | +380 | -320 / +244 |
| `finbert_episode` | -285 | from -1,598 to +834 | +279 | -533 / +248 |

- **None stands out from its reference:** all the intervals contain zero. This holds also for the C of `finbert_episode`: -651 $, interval from -2,202 to +847.
- **All three depend on the same trade,** PYPL-XYZ of 15 July: without that pair they become positive. The pair is not excluded (Step 6).
- **All three lose in the first half of the train and earn in the second.** 13 August does not weigh, and the beta is zero.

The checks do not tell the three candidates apart. For this reason the choice is based on the whole variant (B and C) and on the same entry as A.

**Limits.**
- The bootstrap treats the days as independent.
- The sensitivity to the signal parameters cannot be done for the variants without running the agents again.
- With 21 variants compared, some difference that looks significant is expected by chance.